<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>NN-AE - DIMENSIONALITY REDUCTION + STATIC MAPPING</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import time
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor)

In [ ]:
# Set random seed for reproducibility
seed = 15
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
inp_neuron   = x_dim*y_dim  #Nfile - 1
epochs       = 2000
batchsize    = 256 

learn_rate   = 1e-2
step_epoch   = 120
decay_rate   = 0.750

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_256_0.vtk"
data_file    = "Grid_256_"
Nfile        = 1000                                 # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../../")
directory    = os.getcwd()
path         = directory + "/Flow_Data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale       = 10.0
yscale       = 5.0
vort_max     = 4.4


<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))/yscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname  = 'vort_z'
vort       = np.zeros((Nfile, x_dim, y_dim))

for i in range(Nfile):
    file_name = data_file + str(i) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput() 
    pointData = data.GetPointData().GetArray(fieldname)    
    vort    [i, :, :] = np.reshape(vtk_to_numpy(pointData), (x_dim, y_dim))
    
    print ("*"*85)
vort = vort / vort_max

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS - AE
</div>


In [ ]:
class ENCODER (nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inp_neuron, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )    
    def forward(self,x):
        output = self.encoder(x)
        return output
    
class DECODER (nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, inp_neuron)
        ) 
    
    def forward(self, x):
        output = self.decoder(x)
        return output
    
def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort).to(processor).reshape(vort.shape[0], -1)
vort_output  = torch.Tensor(vort).to(processor).reshape(vort.shape[0], -1)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

dataset               = TensorDataset(vort_input, vort_output)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10,        shuffle=False)
plot_loader    = DataLoader(dataset,    batch_size=batchsize, shuffle=False)
    
encoder_model  = ENCODER ().to(processor)
decoder_model  = DECODER ().to(processor)

encoder_optim  = optim.Adam(encoder_model.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
decoder_optim  = optim.Adam(decoder_model.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)

encoder_sched  = optim.lr_scheduler.StepLR(encoder_optim, step_size=step_epoch, gamma=decay_rate)
decoder_sched  = optim.lr_scheduler.StepLR(decoder_optim, step_size=step_epoch, gamma=decay_rate)

encoder_params = count_params(encoder_model)
decoder_params = count_params(decoder_model)

print(f"Encoder Parameters: {encoder_params}")
print(f"Decoder Parameters: {decoder_params}")


<div style="text-align: center; font-weight: bold; font-size: 1em;">
    AE - TRAINING DATA
</div>


In [ ]:
Loss_Data   = torch.empty(size=(epochs, 1))
loss_func   = nn.MSELoss()

start_time = time.time()

for epoch in range (epochs):
    total_loss = 0.0
    for batch_idx, (w_in, w_out) in enumerate(train_loader):
        encoder_output = encoder_model (w_in)
        decoder_output = decoder_model (encoder_output)
        batch_loss     = loss_func   (decoder_output, w_out)

        encoder_optim.zero_grad()
        decoder_optim.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            encoder_optim.step()
            decoder_optim.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', encoder_optim.param_groups[0]['lr'])
    print ("*"*85)
    
    encoder_sched.step()
    decoder_sched.step()

    
total_time = time.time() - start_time
print(f"\nTotal training wall time: {total_time/3600:.2f} hours")

TOTAL AVERAGE LOSS, [EPOCH = 47 ]
LOSS          : 0.004157236311584711
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 48 ]
LOSS          : 0.004939540289342403
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 49 ]
LOSS          : 0.004894721321761608
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 50 ]
LOSS          : 0.004872472025454044
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 51 ]
LOSS          : 0.0047418903559446335
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 52 ]
LOSS          : 0.0047698128037154675
LEARNING RATE : 0.01
****************

TOTAL AVERAGE LOSS, [EPOCH = 93 ]
LOSS          : 0.0015429682098329067
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 94 ]
LOSS          : 0.0015369667671620846
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 95 ]
LOSS          : 0.001539111603051424
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 96 ]
LOSS          : 0.0015396344242617488
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 97 ]
LOSS          : 0.0015255416510626674
LEARNING RATE : 0.01
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 98 ]
LOSS          : 0.0015355122741311789
LEARNING RATE : 0.01
*************

TOTAL AVERAGE LOSS, [EPOCH = 139 ]
LOSS          : 0.0014921523397788405
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 140 ]
LOSS          : 0.0014808977721258998
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 141 ]
LOSS          : 0.001489040208980441
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 142 ]
LOSS          : 0.0015022812876850367
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 143 ]
LOSS          : 0.0015155987348407507
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 144 ]
LOSS          : 0.0014971881173551083
LEARNING RATE : 0.

TOTAL AVERAGE LOSS, [EPOCH = 185 ]
LOSS          : 0.0010568908182904124
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 186 ]
LOSS          : 0.0010455213487148285
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 187 ]
LOSS          : 0.0011847747955471277
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 188 ]
LOSS          : 0.0010847923113033175
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 189 ]
LOSS          : 0.001053320593200624
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 190 ]
LOSS          : 0.0009487884817644954
LEARNING RATE : 0.

TOTAL AVERAGE LOSS, [EPOCH = 231 ]
LOSS          : 0.0007940471405163407
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 232 ]
LOSS          : 0.0007949593709781766
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 233 ]
LOSS          : 0.000797211891040206
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 234 ]
LOSS          : 0.000778065761551261
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 235 ]
LOSS          : 0.000774975516833365
LEARNING RATE : 0.0075
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 236 ]
LOSS          : 0.0007880686316639185
LEARNING RATE : 0.00

...

TOTAL AVERAGE LOSS, [EPOCH = 1804 ]
LOSS          : 1.0569830919848755e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1805 ]
LOSS          : 1.0189542081207037e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1806 ]
LOSS          : 1.0317948181182146e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1807 ]
LOSS          : 1.0487517101864796e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1808 ]
LOSS          : 1.0426937478769105e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************


TOTAL AVERAGE LOSS, [EPOCH = 1845 ]
LOSS          : 1.0233534339931794e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1846 ]
LOSS          : 1.0189380191150121e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1847 ]
LOSS          : 1.0268945516145322e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1848 ]
LOSS          : 1.0171283065574244e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1849 ]
LOSS          : 1.023809272737708e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
T

TOTAL AVERAGE LOSS, [EPOCH = 1887 ]
LOSS          : 1.010890628094785e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1888 ]
LOSS          : 1.013299697660841e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1889 ]
LOSS          : 1.0088461749546696e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1890 ]
LOSS          : 1.0149236004508566e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1891 ]
LOSS          : 1.0081111213366967e-05
LEARNING RATE : 0.00013363461010158065
*************************************************************************************
TO

TOTAL AVERAGE LOSS, [EPOCH = 1929 ]
LOSS          : 1.000729389488697e-05
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1930 ]
LOSS          : 9.906160812533926e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1931 ]
LOSS          : 9.893685273709707e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1932 ]
LOSS          : 9.722648428578395e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1933 ]
LOSS          : 9.641416909289546e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL

TOTAL AVERAGE LOSS, [EPOCH = 1971 ]
LOSS          : 9.552875781082548e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1972 ]
LOSS          : 9.293128641729709e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1973 ]
LOSS          : 9.540200153423939e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1974 ]
LOSS          : 9.531514479022007e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL AVERAGE LOSS, [EPOCH = 1975 ]
LOSS          : 9.5720879471628e-06
LEARNING RATE : 0.00010022595757618548
*************************************************************************************
TOTAL A

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(encoder_model.state_dict(), "AE_ENCODER_MODEL.pt" )
torch.save(decoder_model.state_dict(), "AE_DECODER_MODEL.pt" )
torch.save(Loss_Data  [0:epoch],  "AE_Loss.pt"  )

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        FNO - TESTING DATA
</div>


In [ ]:
# TEST ERROR OF FNO
encoder_model.eval()
decoder_model.eval()

total_loss = 0.0
for batch_idx, (input_data, output_data) in enumerate(test_loader):
    encoder_output_data = encoder_model (input_data)
    decoder_output_data = decoder_model (encoder_output_data)
    batch_loss          = loss_func (decoder_output_data, output_data)

    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())